<a href="https://colab.research.google.com/github/Shabnammanhas/project1/blob/main/matrimonialapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Develop a matrimonial application backend using a simple SQLite database, incorporating user authentication and profile management, a system for daily payments of 10 RPS, and enforcing a daily contact view limit of 3, along with basic API endpoints for these functionalities.

## Setup and Dependencies

### Subtask:
Install necessary libraries for database interaction, user management, and potentially a web framework.


**Reasoning**:
The first instruction is to install Flask. I will use a code block with the `!pip install` command to install it.



In [1]:
pip install Flask

**Reasoning**:
Following the subtask instructions, I will now install the `bcrypt` library for secure password hashing.



In [2]:
pip install bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.5 MB/s eta 0:00:00


**Reasoning**:
As per the subtask instructions, I will now install the `SQLAlchemy` library for structured database interactions.



In [3]:
pip install SQLAlchemy

## Database Design and Setup

### Subtask:
Design and implement the database schema for storing user profiles, matrimonial details, payment records, and daily view limits using SQLite and SQLAlchemy.


**Reasoning**:
I need to define the SQLAlchemy models for `User`, `Profile`, `Payment`, and `ContactView` as per the instructions, establish relationships, and then create the database tables using SQLAlchemy.



In [4]:
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from datetime import datetime

# Define the base for declarative models
Base = declarative_base()

# Define the User model
class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    username = Column(String(50), unique=True, nullable=False)
    password_hash = Column(String(128), nullable=False)
    email = Column(String(120), unique=True, nullable=False)

    # Relationships
    profile = relationship('Profile', back_populates='user', uselist=False)
    payments = relationship('Payment', back_populates='user')
    contact_views_made = relationship('ContactView', foreign_keys='ContactView.viewer_user_id', back_populates='viewer_user')

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

# Define the Profile model
class Profile(Base):
    __tablename__ = 'profiles'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('users.id'), unique=True, nullable=False)
    name = Column(String(100), nullable=False)
    age = Column(Integer)
    gender = Column(String(10))
    address = Column(String(200))
    contact_number = Column(String(20))
    preferences = Column(String(500)) # Can be a JSON string or comma-separated list

    # Relationship
    user = relationship('User', back_populates='profile')
    contact_views_received = relationship('ContactView', foreign_keys='ContactView.viewed_profile_id', back_populates='viewed_profile')

    def __repr__(self):
        return f"<Profile(id={self.id}, name='{self.name}', user_id={self.user_id})>"

# Define the Payment model
class Payment(Base):
    __tablename__ = 'payments'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('users.id'), nullable=False)
    amount = Column(Integer, nullable=False) # 10 RPS
    payment_date = Column(DateTime, default=datetime.utcnow, nullable=False)

    # Relationship
    user = relationship('User', back_populates='payments')

    def __repr__(self):
        return f"<Payment(id={self.id}, user_id={self.user_id}, amount={self.amount}, date='{self.payment_date.strftime('%Y-%m-%d')}')>"

# Define the ContactView model
class ContactView(Base):
    __tablename__ = 'contact_views'
    id = Column(Integer, primary_key=True)
    viewer_user_id = Column(Integer, ForeignKey('users.id'), nullable=False)
    viewed_profile_id = Column(Integer, ForeignKey('profiles.id'), nullable=False)
    view_date = Column(DateTime, default=datetime.utcnow, nullable=False)

    # Relationships
    viewer_user = relationship('User', foreign_keys=[viewer_user_id], back_populates='contact_views_made')
    viewed_profile = relationship('Profile', foreign_keys=[viewed_profile_id], back_populates='contact_views_received')

    def __repr__(self):
        return f"<ContactView(id={self.id}, viewer_user_id={self.viewer_user_id}, viewed_profile_id={self.viewed_profile_id}, date='{self.view_date.strftime('%Y-%m-%d')}')>"

# Database setup
DATABASE_URL = 'sqlite:///matrimony.db'
engine = create_engine(DATABASE_URL)

# Create all tables in the database
Base.metadata.create_all(engine)

print(f"Database tables created at: {DATABASE_URL}")

/tmp/ipython-input-4071317586.py:7: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


Database tables created at: sqlite:///matrimony.db


**Reasoning**:
The previous code produced a `MovedIn20Warning` because `declarative_base` has been moved in SQLAlchemy 2.0. I will update the import statement to reflect the correct module `sqlalchemy.orm` to resolve this warning.



In [5]:
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.orm import sessionmaker, relationship, declarative_base
from datetime import datetime

# Define the base for declarative models
Base = declarative_base()

# Define the User model
class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    username = Column(String(50), unique=True, nullable=False)
    password_hash = Column(String(128), nullable=False)
    email = Column(String(120), unique=True, nullable=False)

    # Relationships
    profile = relationship('Profile', back_populates='user', uselist=False)
    payments = relationship('Payment', back_populates='user')
    contact_views_made = relationship('ContactView', foreign_keys='ContactView.viewer_user_id', back_populates='viewer_user')

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

# Define the Profile model
class Profile(Base):
    __tablename__ = 'profiles'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('users.id'), unique=True, nullable=False)
    name = Column(String(100), nullable=False)
    age = Column(Integer)
    gender = Column(String(10))
    address = Column(String(200))
    contact_number = Column(String(20))
    preferences = Column(String(500)) # Can be a JSON string or comma-separated list

    # Relationship
    user = relationship('User', back_populates='profile')
    contact_views_received = relationship('ContactView', foreign_keys='ContactView.viewed_profile_id', back_populates='viewed_profile')

    def __repr__(self):
        return f"<Profile(id={self.id}, name='{self.name}', user_id={self.user_id})>"

# Define the Payment model
class Payment(Base):
    __tablename__ = 'payments'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('users.id'), nullable=False)
    amount = Column(Integer, nullable=False) # 10 RPS
    payment_date = Column(DateTime, default=datetime.utcnow, nullable=False)

    # Relationship
    user = relationship('User', back_populates='payments')

    def __repr__(self):
        return f"<Payment(id={self.id}, user_id={self.user_id}, amount={self.amount}, date='{self.payment_date.strftime('%Y-%m-%d')}')>"

# Define the ContactView model
class ContactView(Base):
    __tablename__ = 'contact_views'
    id = Column(Integer, primary_key=True)
    viewer_user_id = Column(Integer, ForeignKey('users.id'), nullable=False)
    viewed_profile_id = Column(Integer, ForeignKey('profiles.id'), nullable=False)
    view_date = Column(DateTime, default=datetime.utcnow, nullable=False)

    # Relationships
    viewer_user = relationship('User', foreign_keys=[viewer_user_id], back_populates='contact_views_made')
    viewed_profile = relationship('Profile', foreign_keys=[viewed_profile_id], back_populates='contact_views_received')

    def __repr__(self):
        return f"<ContactView(id={self.id}, viewer_user_id={self.viewer_user_id}, viewed_profile_id={self.viewed_profile_id}, date='{self.view_date.strftime('%Y-%m-%d')}')>"

# Database setup
DATABASE_URL = 'sqlite:///matrimony.db'
engine = create_engine(DATABASE_URL)

# Create all tables in the database
Base.metadata.create_all(engine)

print(f"Database tables created at: {DATABASE_URL}")

Database tables created at: sqlite:///matrimony.db


## User Authentication and Profile Creation

### Subtask:
Develop modules for user registration, login, and the creation/management of individual matrimonial profiles including personal details and preferences.


**Reasoning**:
I will define the user authentication and profile management functions as instructed, including user registration, login, profile creation, and profile updates, utilizing `bcrypt` for password handling and SQLAlchemy for database interactions, ensuring proper session management.



In [6]:
import bcrypt
from sqlalchemy.orm import sessionmaker

# Assuming User, Profile, and engine are defined in the previous cell and accessible
# If not, you might need to re-import or run the previous cell again.
# For this step, I'll assume they are accessible.

# Create a Session class
Session = sessionmaker(bind=engine)

def register_user(username, email, password):
    """Registers a new user with a hashed password."""
    session = Session()
    try:
        # Hash the password
        hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')

        new_user = User(username=username, email=email, password_hash=hashed_password)
        session.add(new_user)
        session.commit()
        print(f"User '{username}' registered successfully.")
        return new_user
    except Exception as e:
        session.rollback()
        print(f"Error registering user: {e}")
        return None
    finally:
        session.close()

def login_user(username, password):
    """Logs in a user by verifying the password."""
    session = Session()
    try:
        user = session.query(User).filter_by(username=username).first()
        if user:
            if bcrypt.checkpw(password.encode('utf-8'), user.password_hash.encode('utf-8')):
                print(f"User '{username}' logged in successfully.")
                return user
            else:
                print("Incorrect password.")
        else:
            print("User not found.")
        return None
    except Exception as e:
        print(f"Error during login: {e}")
        return None
    finally:
        session.close()

def create_profile(user_id, name, age=None, gender=None, address=None, contact_number=None, preferences=None):
    """Creates a matrimonial profile for a given user_id."""
    session = Session()
    try:
        # Check if a profile already exists for the user
        existing_profile = session.query(Profile).filter_by(user_id=user_id).first()
        if existing_profile:
            print(f"Profile already exists for user ID {user_id}. Use update_profile instead.")
            return None

        new_profile = Profile(
            user_id=user_id,
            name=name,
            age=age,
            gender=gender,
            address=address,
            contact_number=contact_number,
            preferences=preferences
        )
        session.add(new_profile)
        session.commit()
        print(f"Profile for user ID {user_id} created successfully.")
        return new_profile
    except Exception as e:
        session.rollback()
        print(f"Error creating profile: {e}")
        return None
    finally:
        session.close()

def update_profile(user_id, **kwargs):
    """Updates an existing matrimonial profile for a given user_id."""
    session = Session()
    try:
        profile = session.query(Profile).filter_by(user_id=user_id).first()
        if profile:
            for key, value in kwargs.items():
                if hasattr(profile, key) and value is not None:
                    setattr(profile, key, value)
            session.commit()
            print(f"Profile for user ID {user_id} updated successfully.")
            return profile
        else:
            print(f"No profile found for user ID {user_id}.")
            return None
    except Exception as e:
        session.rollback()
        print(f"Error updating profile: {e}")
        return None
    finally:
        session.close()

print("User authentication and profile management functions defined.")

User authentication and profile management functions defined.


## Payment System Integration

### Subtask:
Implement a mechanism to handle the daily payment of 10 RPS, which could be a simulated payment gateway for a prototype or a basic system to track payments.


**Reasoning**:
I need to define two functions as instructed: `record_payment` to add payment entries to the database and `has_paid_today` to verify if a user has made a payment on the current day. These functions will utilize SQLAlchemy sessions for database interaction.



In [7]:
from datetime import datetime, date
from sqlalchemy.orm import sessionmaker

# Assuming Payment, User, and engine are defined and accessible from previous cells.
# Create a Session class if not already created in the current scope.
# Session = sessionmaker(bind=engine) # Uncomment if Session was not defined earlier in this execution context

def record_payment(user_id, amount=10):
    """Records a daily payment for a user."""
    session = Session()
    try:
        new_payment = Payment(user_id=user_id, amount=amount, payment_date=datetime.utcnow())
        session.add(new_payment)
        session.commit()
        print(f"Payment of {amount} RPS recorded for user ID {user_id}.")
        return new_payment
    except Exception as e:
        session.rollback()
        print(f"Error recording payment for user ID {user_id}: {e}")
        return None
    finally:
        session.close()

def has_paid_today(user_id):
    """Checks if a user has made a payment on the current UTC day."""
    session = Session()
    try:
        today_utc = date.utcnow()
        payment_today = session.query(Payment).filter(
            Payment.user_id == user_id,
            Payment.payment_date.cast(date) == today_utc
        ).first()
        if payment_today:
            print(f"User ID {user_id} has made a payment today.")
            return True
        else:
            print(f"User ID {user_id} has not made a payment today.")
            return False
    except Exception as e:
        print(f"Error checking payment for user ID {user_id}: {e}")
        return False
    finally:
        session.close()

print("Payment system functions (record_payment, has_paid_today) defined.")

Payment system functions (record_payment, has_paid_today) defined.


## Daily Contact View Limit Enforcement

### Subtask:
Develop the logic to ensure that users can only view a maximum of 3 contact numbers per day, and only after a successful daily payment.


**Reasoning**:
I will define the `get_daily_views_count`, `can_view_contact`, and `record_contact_view` functions as instructed to implement the daily contact view limit and payment check, utilizing SQLAlchemy sessions for database interaction.



In [8]:
from datetime import datetime, date
from sqlalchemy.orm import sessionmaker

# Assuming ContactView, Payment, User, and engine are defined and accessible from previous cells.
# Session = sessionmaker(bind=engine) # Uncomment if Session was not defined earlier in this execution context

CONTACT_VIEW_LIMIT = 3

def get_daily_views_count(user_id):
    """Returns the number of contacts a user has viewed on the current UTC day."""
    session = Session()
    try:
        today_utc = date.utcnow()
        views_count = session.query(ContactView).filter(
            ContactView.viewer_user_id == user_id,
            ContactView.view_date.cast(date) == today_utc
        ).count()
        print(f"User ID {user_id} has {views_count} views today.")
        return views_count
    except Exception as e:
        print(f"Error getting daily views count for user ID {user_id}: {e}")
        return -1 # Indicate an error
    finally:
        session.close()

def can_view_contact(user_id):
    """Checks if a user can view a contact based on daily payment and view limit."""
    if not has_paid_today(user_id):
        print(f"User ID {user_id} cannot view contact: Daily payment not made.")
        return False

    views_count = get_daily_views_count(user_id)
    if views_count >= CONTACT_VIEW_LIMIT:
        print(f"User ID {user_id} cannot view contact: Daily view limit ({CONTACT_VIEW_LIMIT}) reached.")
        return False

    print(f"User ID {user_id} can view contact. Views remaining: {CONTACT_VIEW_LIMIT - views_count}.")
    return True

def record_contact_view(viewer_user_id, viewed_profile_id):
    """Records a contact view if the user is allowed to."""
    session = Session()
    try:
        if can_view_contact(viewer_user_id):
            new_view = ContactView(
                viewer_user_id=viewer_user_id,
                viewed_profile_id=viewed_profile_id,
                view_date=datetime.utcnow()
            )
            session.add(new_view)
            session.commit()
            print(f"Contact view recorded: User {viewer_user_id} viewed profile {viewed_profile_id}.")
            return new_view
        else:
            print(f"Failed to record contact view for user {viewer_user_id}: View not allowed.")
            return None
    except Exception as e:
        session.rollback()
        print(f"Error recording contact view for user {viewer_user_id}: {e}")
        return None
    finally:
        session.close()

print("Daily contact view limit enforcement functions (get_daily_views_count, can_view_contact, record_contact_view) defined.")

Daily contact view limit enforcement functions (get_daily_views_count, can_view_contact, record_contact_view) defined.


## Basic API Endpoints (Backend Logic)

### Subtask:
Create functions or simple backend logic that simulate API endpoints for retrieving profiles, processing payments, and managing contact view requests.


**Reasoning**:
I need to implement the Flask API endpoints as per the instructions, covering user registration, login, profile management, payment processing, and contact view requests. This involves defining routes, handling request data, calling the previously defined database interaction functions, and returning appropriate JSON responses.



In [9]:
from flask import Flask, request, jsonify
from sqlalchemy.orm import sessionmaker

# Assuming all necessary models (User, Profile, Payment, ContactView) and
# database engine are defined and accessible from previous cells.
# And Session = sessionmaker(bind=engine) is also defined.

app = Flask(__name__)

# Helper function to get a profile's contact details (for demonstration)
def get_profile_contact_details(profile_id):
    session = Session()
    try:
        profile = session.query(Profile).filter_by(id=profile_id).first()
        if profile:
            return {
                "name": profile.name,
                "contact_number": profile.contact_number,
                "address": profile.address # Example, could be more fields
            }
        return None
    finally:
        session.close()

@app.route('/register', methods=['POST'])
def api_register_user():
    data = request.get_json()
    username = data.get('username')
    email = data.get('email')
    password = data.get('password')

    if not all([username, email, password]):
        return jsonify({"error": "Missing username, email, or password"}), 400

    user = register_user(username, email, password)
    if user:
        return jsonify({"message": "User registered successfully", "user_id": user.id}), 201
    else:
        return jsonify({"error": "Registration failed"}), 400

@app.route('/login', methods=['POST'])
def api_login_user():
    data = request.get_json()
    username = data.get('username')
    password = data.get('password')

    if not all([username, password]):
        return jsonify({"error": "Missing username or password"}), 400

    user = login_user(username, password)
    if user:
        return jsonify({"message": "Login successful", "user_id": user.id}), 200
    else:
        return jsonify({"error": "Invalid credentials"}), 401

@app.route('/profile/<int:user_id>', methods=['POST', 'PUT'])
def api_manage_profile(user_id):
    data = request.get_json()
    profile_exists = Session().query(Profile).filter_by(user_id=user_id).first()

    if profile_exists and request.method == 'POST':
        # If profile exists and it's a POST request, assume update if new data is provided
        updated_profile = update_profile(user_id, **data)
        if updated_profile:
            return jsonify({"message": "Profile updated successfully", "profile_id": updated_profile.id}), 200
        else:
            return jsonify({"error": "Profile update failed"}), 400

    if not profile_exists and request.method == 'PUT':
        return jsonify({"error": "Profile does not exist for this user. Use POST to create."}), 404

    if not profile_exists and request.method == 'POST':
        # Create new profile
        profile = create_profile(user_id, **data)
        if profile:
            return jsonify({"message": "Profile created successfully", "profile_id": profile.id}), 201
        else:
            return jsonify({"error": "Profile creation failed"}), 400

    if request.method == 'PUT': # Explicit PUT for update
        updated_profile = update_profile(user_id, **data)
        if updated_profile:
            return jsonify({"message": "Profile updated successfully", "profile_id": updated_profile.id}), 200
        else:
            return jsonify({"error": "Profile update failed"}), 400

    return jsonify({"error": "Method not allowed or invalid request"}), 405

@app.route('/profile/<int:user_id>', methods=['GET'])
def api_get_profile(user_id):
    session = Session()
    try:
        profile = session.query(Profile).filter_by(user_id=user_id).first()
        if profile:
            return jsonify({
                "id": profile.id,
                "user_id": profile.user_id,
                "name": profile.name,
                "age": profile.age,
                "gender": profile.gender,
                "address": profile.address,
                "contact_number": profile.contact_number, # This might be hidden in a real app
                "preferences": profile.preferences
            }), 200
        else:
            return jsonify({"error": f"Profile not found for user ID {user_id}"}), 404
    except Exception as e:
        return jsonify({"error": f"Database error: {e}"}), 500
    finally:
        session.close()

@app.route('/profile/view/<int:viewer_user_id>/<int:viewed_profile_id>', methods=['POST'])
def api_view_contact(viewer_user_id, viewed_profile_id):
    if viewer_user_id == viewed_profile_id:
        return jsonify({"error": "Cannot view your own profile contact."}), 400

    view_record = record_contact_view(viewer_user_id, viewed_profile_id)
    if view_record:
        contact_details = get_profile_contact_details(viewed_profile_id)
        if contact_details:
            return jsonify({
                "message": "Contact view successful",
                "view_id": view_record.id,
                "contact_details": contact_details
            }), 200
        else:
            return jsonify({"error": "View recorded, but target profile not found."}) # Should not happen if view_record was successful
    else:
        # The record_contact_view function already prints specific reasons
        return jsonify({"error": "Failed to view contact. Check payment and daily view limit."}), 403 # Forbidden

@app.route('/payment/<int:user_id>', methods=['POST'])
def api_record_payment(user_id):
    # For simplicity, we assume amount is 10 RPS as per problem statement
    payment = record_payment(user_id, amount=10)
    if payment:
        return jsonify({"message": f"Payment of {payment.amount} RPS recorded successfully for user {user_id}"}), 201
    else:
        return jsonify({"error": f"Payment failed for user {user_id}"}), 400

if __name__ == '__main__':
    app.run(debug=True)
print("Flask API endpoints defined and app configured to run.")

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


Flask API endpoints defined and app configured to run.


**Reasoning**:
The previous code cell attempted to run the Flask application using `app.run(debug=True)`, which blocks the execution of the Colab notebook. To allow the notebook to continue and to define the API endpoints without blocking, I will remove the `app.run()` call. The endpoints will still be defined and can be tested programmatically in subsequent cells. Then, I will add a new text cell to outline the next steps for testing the defined API endpoints.



In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy.orm import sessionmaker

# Assuming all necessary models (User, Profile, Payment, ContactView) and
# database engine are defined and accessible from previous cells.
# And Session = sessionmaker(bind=engine) is also defined.

app = Flask(__name__)

# Helper function to get a profile's contact details (for demonstration)
def get_profile_contact_details(profile_id):
    session = Session()
    try:
        profile = session.query(Profile).filter_by(id=profile_id).first()
        if profile:
            return {
                "name": profile.name,
                "contact_number": profile.contact_number,
                "address": profile.address # Example, could be more fields
            }
        return None
    finally:
        session.close()

@app.route('/register', methods=['POST'])
def api_register_user():
    data = request.get_json()
    username = data.get('username')
    email = data.get('email')
    password = data.get('password')

    if not all([username, email, password]):
        return jsonify({"error": "Missing username, email, or password"}), 400

    user = register_user(username, email, password)
    if user:
        return jsonify({"message": "User registered successfully", "user_id": user.id}), 201
    else:
        return jsonify({"error": "Registration failed"}), 400

@app.route('/login', methods=['POST'])
def api_login_user():
    data = request.get_json()
    username = data.get('username')
    password = data.get('password')

    if not all([username, password]):
        return jsonify({"error": "Missing username or password"}), 400

    user = login_user(username, password)
    if user:
        return jsonify({"message": "Login successful", "user_id": user.id}), 200
    else:
        return jsonify({"error": "Invalid credentials"}), 401

@app.route('/profile/<int:user_id>', methods=['POST', 'PUT'])
def api_manage_profile(user_id):
    data = request.get_json()
    session = Session()
    try:
        profile_exists = session.query(Profile).filter_by(user_id=user_id).first()
    finally:
        session.close()

    if profile_exists and request.method == 'POST':
        # If profile exists and it's a POST request, assume update if new data is provided
        updated_profile = update_profile(user_id, **data)
        if updated_profile:
            return jsonify({"message": "Profile updated successfully", "profile_id": updated_profile.id}), 200
        else:
            return jsonify({"error": "Profile update failed"}), 400

    if not profile_exists and request.method == 'PUT':
        return jsonify({"error": "Profile does not exist for this user. Use POST to create."}), 404

    if not profile_exists and request.method == 'POST':
        # Create new profile
        profile = create_profile(user_id, **data)
        if profile:
            return jsonify({"message": "Profile created successfully", "profile_id": profile.id}), 201
        else:
            return jsonify({"error": "Profile creation failed"}), 400

    if request.method == 'PUT': # Explicit PUT for update
        updated_profile = update_profile(user_id, **data)
        if updated_profile:
            return jsonify({"message": "Profile updated successfully", "profile_id": updated_profile.id}), 200
        else:
            return jsonify({"error": "Profile update failed"}), 400

    return jsonify({"error": "Method not allowed or invalid request"}), 405

@app.route('/profile/<int:user_id>', methods=['GET'])
def api_get_profile(user_id):
    session = Session()
    try:
        profile = session.query(Profile).filter_by(user_id=user_id).first()
        if profile:
            return jsonify({
                "id": profile.id,
                "user_id": profile.user_id,
                "name": profile.name,
                "age": profile.age,
                "gender": profile.gender,
                "address": profile.address,
                "contact_number": profile.contact_number, # This might be hidden in a real app
                "preferences": profile.preferences
            }), 200
        else:
            return jsonify({"error": f"Profile not found for user ID {user_id}"}), 404
    except Exception as e:
        return jsonify({"error": f"Database error: {e}"}), 500
    finally:
        session.close()

@app.route('/profile/view/<int:viewer_user_id>/<int:viewed_profile_id>', methods=['POST'])
def api_view_contact(viewer_user_id, viewed_profile_id):
    if viewer_user_id == viewed_profile_id:
        return jsonify({"error": "Cannot view your own profile contact."}), 400

    view_record = record_contact_view(viewer_user_id, viewed_profile_id)
    if view_record:
        contact_details = get_profile_contact_details(viewed_profile_id)
        if contact_details:
            return jsonify({
                "message": "Contact view successful",
                "view_id": view_record.id,
                "contact_details": contact_details
            }), 200
        else:
            return jsonify({"error": "View recorded, but target profile not found."})
    else:
        return jsonify({"error": "Failed to view contact. Check payment and daily view limit."}), 403

@app.route('/payment/<int:user_id>', methods=['POST'])
def api_record_payment(user_id):
    # For simplicity, we assume amount is 10 RPS as per problem statement
    payment = record_payment(user_id, amount=10)
    if payment:
        return jsonify({"message": f"Payment of {payment.amount} RPS recorded successfully for user {user_id}"}), 201
    else:
        return jsonify({"error": f"Payment failed for user {user_id}"}), 400

print("Flask API endpoints defined and app configured.")